
# Second-Scorer Robustness for Context Utility

This notebook tests whether the Ghostbuster context-utility result depends strongly on the choice of independent scorer.

Primary paper scorer:
- `Qwen/Qwen3-0.6B-Base`

Second scorer:
- `Qwen/Qwen3-1.7B-Base`

The measurement definition is kept fixed:

\[
U(k) = L_8 - L_k,\qquad k\in\{32,128,256\}
\]

with:
- first 384 scorer tokens,
- no special tokens,
- eight deterministic target positions,
- exact preceding \(k\) scorer tokens for each target.

To keep the robustness run inexpensive, we use a deterministic subset of complete seven-way Ghostbuster groups and compare:

1. pooled matched AI-human \(\Delta U(k)\) under the second scorer;
2. the same pooled result under the primary scorer on the exact same subset;
3. document-level correlation between scorers;
4. whether the qualitative scale ordering survives.

This is a robustness analysis, not a replacement for the frozen primary scorer.


In [1]:

from pathlib import Path
import gc
import json
import math
import time
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

HERE = Path.cwd().resolve()

candidates = [HERE, HERE.parent, HERE.parent.parent]

REPRO_ROOT = None
for p in candidates:
    if (p / "data" / "raw" / "ghostbuster_essays.parquet").exists():
        REPRO_ROOT = p
        break
    if (p / "reproduction" / "data" / "raw" / "ghostbuster_essays.parquet").exists():
        REPRO_ROOT = p / "reproduction"
        break

if REPRO_ROOT is None:
    raise FileNotFoundError("Could not locate reproduction repository.")

RAW_PATH = REPRO_ROOT / "data" / "raw" / "ghostbuster_essays.parquet"
PRIMARY_PATH = REPRO_ROOT / "results" / "ghostbuster_essay_context.parquet"
OUT_DIR = REPRO_ROOT / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_SCORER = "Qwen/Qwen3-0.6B-Base"
SECOND_SCORER = "Qwen/Qwen3-1.7B-Base"

CONTEXT_LENGTHS = [8, 32, 128, 256]
PREFIX_LEN = 384
N_TARGETS = 8

# Deterministic robustness subset.
N_GROUPS = 150
SUBSET_SEED = 1729

print("REPRO_ROOT:", REPRO_ROOT)
print("Second scorer:", SECOND_SCORER)
print("Target complete groups:", N_GROUPS)


/home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


REPRO_ROOT: ..
Second scorer: Qwen/Qwen3-1.7B-Base
Target complete groups: 150



## 1. Load corrected Ghostbuster corpus and primary-scorer features


In [2]:

raw = pd.read_parquet(RAW_PATH)
primary = pd.read_parquet(PRIMARY_PATH)

print("raw:", raw.shape)
print("primary:", primary.shape)
print("raw columns:", raw.columns.tolist())
print("primary columns:", primary.columns.tolist())


raw: (7000, 8)
primary: (7000, 15)
raw columns: ['row_id', 'file', 'domain', 'generator', 'label', 'group_id', 'text', 'text_sha256']
primary columns: ['row_id', 'eligible_384', 'n_scorer_tokens', 'n_targets', 'context_l_8', 'context_l_16', 'context_l_32', 'context_l_64', 'context_l_128', 'context_l_256', 'context_u_16', 'context_u_32', 'context_u_64', 'context_u_128', 'context_u_256']


In [3]:

def pick_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of {candidates} found in {df.columns.tolist()}")
    return None

group_col = pick_col(raw, ["group_id", "source_id", "essay_id", "original_id"])
gen_col = pick_col(raw, ["generator", "source", "model", "label"])
row_raw = pick_col(raw, ["row_id", "id", "doc_id"])
text_col = pick_col(raw, ["text", "document", "content"])

row_primary = pick_col(primary, ["row_id", "id", "doc_id"])
elig_col = pick_col(primary, ["eligible_384", "eligible", "context_eligible", "is_eligible"])

u_primary = {}
for k in [32, 128, 256]:
    u_primary[k] = pick_col(
        primary,
        [f"context_u_{k}", f"u_{k}", f"U_{k}", f"context_U_{k}"]
    )

base = raw[[group_col, gen_col, row_raw, text_col]].copy().rename(columns={
    group_col: "group_id",
    gen_col: "generator",
    row_raw: "row_id",
    text_col: "text",
})

pcols = [row_primary, elig_col] + list(u_primary.values())
p = primary[pcols].copy().rename(columns={row_primary: "row_id"})
p = p.rename(columns={u_primary[k]: f"primary_U{k}" for k in u_primary})

df = base.merge(p, on="row_id", validate="one_to_one")
df["primary_eligible"] = df[elig_col].astype(bool)

print(df.shape)
display(df.head())


(7000, 9)


,group_id,generator,row_id,text,eligible_384,primary_U32,primary_U128,primary_U256,primary_eligible
0,1,human,essay::human::1,The world has been shattered by Russia’s attac...,False,NaN,NaN,NaN,False
1,1,gpt,essay::gpt::1,"Introduction:\nThe film ""12 Years a Slave"" ser...",True,1.889741,2.329772,2.878357,True
2,1,claude,essay::claude::1,Social class had a profound influence on how w...,True,0.409851,0.816566,0.880846,True
3,1,gpt_prompt1,essay::gpt_prompt1::1,Introduction:\nThe film 12 Years a Slave delve...,True,3.158160,3.315095,3.544277,True
4,1,gpt_prompt2,essay::gpt_prompt2::1,Introduction:\nThe film 12 Years a Slave serve...,True,0.306538,2.912520,3.187547,True



## 2. Select deterministic complete seven-way subset

We sample only from the 708 groups eligible under the frozen primary scorer, ensuring the robustness comparison uses the same long-document regime.


In [4]:

group_ok = (
    df.groupby("group_id")["primary_eligible"]
    .agg(["sum", "count"])
)

complete_ids = group_ok.index[
    (group_ok["sum"] == 7) & (group_ok["count"] == 7)
].to_numpy()

print("complete primary-eligible groups:", len(complete_ids))
assert len(complete_ids) == 708

rng = np.random.default_rng(SUBSET_SEED)
subset_ids = np.sort(
    rng.choice(
        complete_ids,
        size=min(N_GROUPS, len(complete_ids)),
        replace=False,
    )
)

subset = df[df["group_id"].isin(subset_ids)].copy()
subset = subset.sort_values(["group_id", "generator"]).reset_index(drop=True)

print("selected groups:", subset["group_id"].nunique())
print("selected documents:", len(subset))
assert len(subset) == len(subset_ids) * 7

display(
    subset.groupby("generator").size().rename("n").to_frame()
)


complete primary-eligible groups: 708
selected groups: 150
selected documents: 1050


,n
generator,
claude,150
gpt,150
gpt_prompt1,150
gpt_prompt2,150
gpt_semantic,150
gpt_writing,150
human,150



## 3. Exact scorer implementation

The target-position rule matches the recovered Ghostbuster implementation:

- `start = max(context_lengths) + 1 = 257`
- eight positions linearly spaced through the 384-token prefix
- exact preceding \(k\) tokens for each target


In [5]:

def choose_target_positions(prefix_len, context_lengths=CONTEXT_LENGTHS, n_targets=N_TARGETS):
    max_context = max(context_lengths)
    start = max_context + 1
    if start >= prefix_len:
        return []

    positions = np.linspace(
        start,
        prefix_len - 1,
        num=min(n_targets, prefix_len - start),
        dtype=int,
    )
    return np.unique(positions).tolist()

TARGET_POSITIONS = choose_target_positions(PREFIX_LEN)
print("target positions:", TARGET_POSITIONS)
assert len(TARGET_POSITIONS) == 8


target positions: [257, 275, 293, 311, 329, 347, 365, 383]


In [6]:

tokenizer = AutoTokenizer.from_pretrained(
    SECOND_SCORER,
    use_fast=True,
)

model = AutoModelForCausalLM.from_pretrained(
    SECOND_SCORER,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
)

model.eval()

print("loaded:", SECOND_SCORER)
print("device:", next(model.parameters()).device)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████| 310/310 [00:00<00:00, 319.66it/s]


loaded: Qwen/Qwen3-1.7B-Base
device: cuda:0


In [7]:

@torch.inference_mode()
def context_curve_for_text(text):
    token_ids = tokenizer(
        text,
        add_special_tokens=False,
    )["input_ids"]

    if len(token_ids) < PREFIX_LEN:
        return None

    token_ids = token_ids[:PREFIX_LEN]

    losses = {k: [] for k in CONTEXT_LENGTHS}

    # Batch all context windows for one target position.
    for t in TARGET_POSITIONS:
        seqs = []
        targets = []

        for k in CONTEXT_LENGTHS:
            context = token_ids[t-k:t]
            target = token_ids[t]

            seqs.append(torch.tensor(context, dtype=torch.long))
            targets.append(target)

        # Context lengths differ, so score each exact suffix independently.
        # Only 4 windows × 8 targets per document; this stays simple and exact.
        for k, seq, target in zip(CONTEXT_LENGTHS, seqs, targets):
            input_ids = seq.unsqueeze(0).to(model.device)

            out = model(
                input_ids=input_ids,
                use_cache=False,
            )

            logits = out.logits[:, -1, :].float()
            logp = torch.log_softmax(logits, dim=-1)[0, target]
            losses[k].append(float(-logp.cpu()))

    L = {k: float(np.mean(v)) for k, v in losses.items()}

    return {
        "L8": L[8],
        "L32": L[32],
        "L128": L[128],
        "L256": L[256],
        "U32": L[8] - L[32],
        "U128": L[8] - L[128],
        "U256": L[8] - L[256],
    }



## 4. Score the robustness subset

The notebook checkpoints every 100 documents so an interrupted run can be resumed.


In [8]:

CHECKPOINT = OUT_DIR / "ghostbuster_second_scorer_checkpoint.parquet"

if CHECKPOINT.exists():
    scored = pd.read_parquet(CHECKPOINT)
    done = set(scored["row_id"])
    print("resuming from checkpoint:", len(scored))
else:
    scored = pd.DataFrame()
    done = set()

rows = [] if scored.empty else scored.to_dict("records")

t0 = time.time()

for i, row in subset.iterrows():
    if row["row_id"] in done:
        continue

    curve = context_curve_for_text(row["text"])

    if curve is None:
        rec = {
            "row_id": row["row_id"],
            "group_id": row["group_id"],
            "generator": row["generator"],
            "second_eligible": False,
            "second_U32": np.nan,
            "second_U128": np.nan,
            "second_U256": np.nan,
        }
    else:
        rec = {
            "row_id": row["row_id"],
            "group_id": row["group_id"],
            "generator": row["generator"],
            "second_eligible": True,
            "second_U32": curve["U32"],
            "second_U128": curve["U128"],
            "second_U256": curve["U256"],
        }

    rows.append(rec)

    if len(rows) % 25 == 0:
        elapsed = time.time() - t0
        print(f"scored {len(rows)}/{len(subset)} docs; elapsed {elapsed/60:.1f} min")

    if len(rows) % 100 == 0:
        pd.DataFrame(rows).to_parquet(CHECKPOINT, index=False)

second = pd.DataFrame(rows)
second.to_parquet(CHECKPOINT, index=False)

print("done:", second.shape)
display(second.head())


scored 25/1050 docs; elapsed 0.4 min
scored 50/1050 docs; elapsed 0.8 min
scored 75/1050 docs; elapsed 1.3 min
scored 100/1050 docs; elapsed 1.7 min
scored 125/1050 docs; elapsed 2.1 min
scored 150/1050 docs; elapsed 2.5 min
scored 175/1050 docs; elapsed 2.9 min
scored 200/1050 docs; elapsed 3.3 min
scored 225/1050 docs; elapsed 3.8 min
scored 250/1050 docs; elapsed 4.2 min
scored 275/1050 docs; elapsed 4.6 min
scored 300/1050 docs; elapsed 5.0 min
scored 325/1050 docs; elapsed 5.5 min
scored 350/1050 docs; elapsed 5.9 min
scored 375/1050 docs; elapsed 6.4 min
scored 400/1050 docs; elapsed 6.8 min
scored 425/1050 docs; elapsed 7.3 min
scored 450/1050 docs; elapsed 7.7 min
scored 475/1050 docs; elapsed 8.2 min
scored 500/1050 docs; elapsed 8.6 min
scored 525/1050 docs; elapsed 9.1 min
scored 550/1050 docs; elapsed 9.5 min
scored 575/1050 docs; elapsed 10.0 min
scored 600/1050 docs; elapsed 10.4 min
scored 625/1050 docs; elapsed 10.8 min
scored 650/1050 docs; elapsed 11.3 min
scored 675/

,row_id,group_id,generator,second_eligible,second_U32,second_U128,second_U256
0,essay::claude::105,105,claude,True,1.182598,1.467849,1.511055
1,essay::gpt::105,105,gpt,True,0.646577,0.967075,1.678408
2,essay::gpt_prompt1::105,105,gpt_prompt1,True,0.827342,0.931271,1.261610
3,essay::gpt_prompt2::105,105,gpt_prompt2,True,0.068916,0.946741,0.747082
4,essay::gpt_semantic::105,105,gpt_semantic,True,1.185016,1.884861,1.869044



## 5. Compare scorers on the exact same documents


In [10]:

cmp = subset[
    [
        "row_id", "group_id", "generator",
        "primary_U32", "primary_U128", "primary_U256"
    ]
].merge(
    second,
    on=["row_id", "group_id", "generator"],
    validate="one_to_one",
)

print("second scorer eligible:", int(cmp["second_eligible"].sum()), "/", len(cmp))

for k in [32, 128, 256]:
    finite = cmp[[f"primary_U{k}", f"second_U{k}"]].dropna()
    corr = finite.corr().iloc[0, 1]
    mad = (finite[f"primary_U{k}"] - finite[f"second_U{k}"]).abs().mean()

    print(
        f"k={k}: n={len(finite)}, "
        f"corr={corr:.4f}, MAD={mad:.4f}"
    )


second scorer eligible: 1050 / 1050
k=32: n=1050, corr=0.8553, MAD=0.2742
k=128: n=1050, corr=0.9047, MAD=0.2770
k=256: n=1050, corr=0.9121, MAD=0.2818



## 6. Matched AI-human effect under each scorer

For comparability, require all seven documents to be eligible under the second scorer, then compute both primary- and second-scorer estimates on that exact same subset.


In [11]:

second_group_ok = (
    cmp.groupby("group_id")["second_eligible"]
    .agg(["sum", "count"])
)

common_ids = second_group_ok.index[
    (second_group_ok["sum"] == 7)
    & (second_group_ok["count"] == 7)
]

common = cmp[cmp["group_id"].isin(common_ids)].copy()

print("common complete groups:", len(common_ids))

machine_generators = sorted(
    [g for g in common["generator"].unique() if g != "human"]
)

def pooled_matched_delta(data, prefix, k):
    wide = data.pivot(
        index="group_id",
        columns="generator",
        values=f"{prefix}_U{k}",
    )

    machine_mean = wide[machine_generators].mean(axis=1)
    return machine_mean - wide["human"]

def bootstrap_ci(x, n_boot=5000, seed=42):
    x = np.asarray(x, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(n_boot, len(x)))
    means = x[idx].mean(axis=1)
    lo, hi = np.quantile(means, [0.025, 0.975])
    return x.mean(), lo, hi

summary_rows = []

for k in [32, 128, 256]:
    for scorer in ["primary", "second"]:
        delta = pooled_matched_delta(common, scorer, k)
        mean, lo, hi = bootstrap_ci(delta, seed=1000 + k + (0 if scorer == "primary" else 1))

        summary_rows.append({
            "scorer": scorer,
            "k": k,
            "n_groups": len(delta),
            "mean_delta": mean,
            "ci_low": lo,
            "ci_high": hi,
        })

summary = pd.DataFrame(summary_rows)
display(summary)


common complete groups: 150


,scorer,k,n_groups,mean_delta,ci_low,ci_high
0,primary,32,150,0.267407,0.166265,0.368554
1,second,32,150,0.273521,0.176279,0.375198
2,primary,128,150,0.345617,0.214903,0.477219
3,second,128,150,0.333962,0.196347,0.468400
4,primary,256,150,0.389575,0.245969,0.529301
5,second,256,150,0.389691,0.251988,0.525785



## 7. Scorer-level robustness summary


In [12]:

pivot = summary.pivot(index="k", columns="scorer", values="mean_delta")
pivot["second_minus_primary"] = pivot["second"] - pivot["primary"]

display(pivot)

second_only = summary[summary["scorer"] == "second"].sort_values("k")

all_positive = (second_only["mean_delta"] > 0).all()
all_ci_positive = (second_only["ci_low"] > 0).all()

print("Second-scorer point estimates all positive:", all_positive)
print("Second-scorer CIs all above zero:", all_ci_positive)

vals = second_only["mean_delta"].to_numpy()
nondecreasing = np.all(np.diff(vals) >= 0)

print("Second-scorer pooled ΔU nondecreasing with k:", nondecreasing)

if all_positive and nondecreasing:
    print(
        "\nInterpretation: the qualitative Ghostbuster context-utility result "
        "survives a larger independent scorer."
    )
else:
    print(
        "\nInterpretation: scorer choice materially changes at least one aspect "
        "of the Ghostbuster context-utility profile; inspect before paper use."
    )


scorer,primary,second,second_minus_primary
k,,,
32,0.267407,0.273521,0.006113
128,0.345617,0.333962,-0.011656
256,0.389575,0.389691,0.000116


Second-scorer point estimates all positive: True
Second-scorer CIs all above zero: True
Second-scorer pooled ΔU nondecreasing with k: True

Interpretation: the qualitative Ghostbuster context-utility result survives a larger independent scorer.



## 8. Save results


In [13]:

doc_path = OUT_DIR / "ghostbuster_second_scorer_document_comparison.parquet"
summary_path = OUT_DIR / "ghostbuster_second_scorer_summary.csv"

cmp.to_parquet(doc_path, index=False)
summary.to_csv(summary_path, index=False)

print(doc_path)
print(summary_path)


../results/ghostbuster_second_scorer_document_comparison.parquet
../results/ghostbuster_second_scorer_summary.csv
